# Notebook 6 — Evaluation & Comparison Summary

Evaluation is the most critical phase of any machine learning project — it determines whether our models are actually useful beyond just fitting training data. Throughout the project we used RMSE and MAE to measure rating prediction accuracy. While these metrics confirm the model can predict ratings, they don't answer the real question: are we recommending the right movies? A model could predict ratings accurately but still recommend movies the user doesn't care about.

This notebook introduces ranking metrics which measure recommendation quality directly. Precision@K asks what fraction of our top K recommendations the user actually liked. Recall@K asks what fraction of all movies the user liked did we successfully surface. NDCG@K asks whether we ranked the movies the user liked near the top of our list — reflecting the reality that users rarely scroll past the first few recommendations.

This notebook evaluates all four models uniformly on the same 100 sampled users using the same ground truth for the first time. The key steps include:

- Loading all saved models, encoders and artifacts from previous phases  
- Defining Precision@K, Recall@K and NDCG@K metric functions  
- Defining individual scoring functions for all four models  
- Sampling 100 users present in both train and test sets with fixed random seed (42) for reproducibility and fair comparison  
- Evaluating all four models on identical users with identical ground truth  
- Comparing original hybrid weights (SVD = 0.4, NCF = 0.4, Content = 0.2) against tuned weights (SVD = 0.35, NCF = 0.50, Content = 0.15)  
- Saving updated hybrid config and final evaluation results  

---

## Final Evaluation Results @ K=10

| Model           | RMSE   | MAE    | P@10  | R@10  | NDCG@10 |
|:----------------|:------:|:------:|:-----:|:-----:|:-------:|
| SVD             | 0.9372 | 0.7440 | 0.0837 | 0.0167 | 0.0747 |
| Neural CF       | 0.9587 | 0.7598 | 0.0980 | 0.0208 | 0.0970 |
| Content Based   | N/A    | N/A    | 0.0153 | 0.0032 | 0.0186 |
| Hybrid (Tuned)  | N/A    | N/A    | 0.1041 | 0.0277 | 0.1018 |

---

## Key Findings

- **Hybrid (Tuned)** outperforms all individual models on every ranking metric confirming the combination strategy is effective  
- **NCF** outperforms **SVD** on all ranking metrics despite SVD having better RMSE — demonstrating that rating prediction accuracy does not equal recommendation quality  
- **Content Based** is the weakest model on ranking metrics justifying its lower weight in the hybrid  
- Tuned weights (**SVD = 0.35, NCF = 0.50, Content = 0.15**) outperform original weights with average score improving from **0.0750 to 0.0779**  
- All ranking metrics are in the expected range for recommendation systems on explicit rating datasets — Precision@10 of ~0.10 means roughly 1 in 10 recommendations is something the user rates highly which is csistent with published literature  

In [1]:
import pandas as pd                          # data manipulation
import numpy as np                           # numerical operations
import pickle                                # loading saved models
import os                                    # file operations
from surprise import SVD                     # SVD collaborative filtering
from tensorflow import keras                 # NCF deep learning model
from sklearn.preprocessing import MinMaxScaler  # score normalization for hybrid
import json                                  # loading hybrid config file
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

All libraries loaded successfully!


# Load Data and Models

In [2]:
base_path = "OneDrive/Documents/Recommendation System"
models_path = f"{base_path}/models"

# Load processed train and test files from Phase 1
# Train — what users rated in the past (model input)
# Test — what users rated later (ground truth for evaluation)
train = pd.read_csv(f'{base_path}/train.csv')
test = pd.read_csv(f'{base_path}/test.csv')
movies = pd.read_csv(f'{base_path}/movies.csv')

print("Data loaded!")
print(f"Train: {train.shape}, Test: {test.shape}, Movies: {movies.shape}")

# Load SVD model — saved as pickle in Phase 3
with open(f'{models_path}/svd_model.pkl', 'rb') as f:
    svd_model = pickle.load(f)
print("SVD model loaded!")

# Load NCF model — saved in native keras format in Phase 5
ncf_model = keras.models.load_model(f'{models_path}/ncf_model.keras')

# Load encoders — must be identical to what was used during NCF training
# Using different encoders would map IDs to wrong indices
with open(f'{models_path}/user_encoder.pkl', 'rb') as f:
    user_encoder = pickle.load(f)
with open(f'{models_path}/movie_encoder.pkl', 'rb') as f:
    movie_encoder = pickle.load(f)
print("NCF model and encoders loaded!")

# Load cosine similarity matrix computed in Phase 4
# Shape is (3883, 3883) — similarity between every pair of movies
cosine_sim = np.load(f'{models_path}/cosine_sim.npy')

# Load movie title to index mapping for content based lookups
movie_indices = pd.read_pickle(f'{models_path}/movie_indices.pkl')
print("Content based models loaded!")

# Load hybrid weights configuration saved in Phase 6
with open(f'{base_path}/hybrid_config.json', 'r') as f:
    hybrid_config = json.load(f)
print("Hybrid config loaded:", hybrid_config)

print("\nAll models and data loaded successfully!")

Data loaded!
Train: (800167, 11), Test: (200042, 11), Movies: (3883, 3)
SVD model loaded!
NCF model and encoders loaded!
Content based models loaded!
Hybrid config loaded: {'svd_weight': 0.4, 'ncf_weight': 0.4, 'content_weight': 0.2}

All models and data loaded successfully!


# Genre Preprocessing

In [3]:
# Recreate exact same genre cleaning from Phase 4
# Content based scoring relies on these cleaned genre strings
# Must match exactly what was used to build the cosine similarity matrix

# Replace pipe separator with space so each genre is a separate token
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

# Replace hyphenated genres with single tokens
# Sci-Fi → SciFi prevents TF-IDF splitting into 'sci' and 'fi'
movies['genres_clean'] = movies['genres_clean'].str.replace('Sci-Fi', 'SciFi', regex=False)

# Film-Noir → FilmNoir prevents splitting into 'film' and 'noir'
movies['genres_clean'] = movies['genres_clean'].str.replace('Film-Noir', 'FilmNoir', regex=False)

# Children's → Childrens removes apostrophe that confuses tokenizers
movies['genres_clean'] = movies['genres_clean'].str.replace("Children's", 'Childrens', regex=False)

# Fill any missing genres with empty string to avoid NaN errors
movies['genres_clean'] = movies['genres_clean'].fillna('')

print("Genre preprocessing complete!")

Genre preprocessing complete!


# Define Evaluation Metrics

In [4]:
def precision_at_k(recommended, relevant, k):
    """
    Precision@K: fraction of top-K recommendations that are relevant.
    
    Question: of the K movies we recommended, what fraction did user actually like?
    Example: recommend 10 movies, user liked 3 → Precision@10 = 0.3
    Higher is better, max value = 1.0
    """
    # Consider only top K recommendations
    top_k = recommended[:k]
    
    # Count how many of top K recommendations user actually liked
    hits = len([m for m in top_k if m in relevant])
    
    # Divide hits by K to get fraction
    return hits / k if k > 0 else 0.0


def recall_at_k(recommended, relevant, k):
    """
    Recall@K: fraction of relevant movies that appear in top-K recommendations.
    
    Question: of all movies user liked, how many did we successfully recommend?
    Example: user liked 20 movies, we recommended 5 of them → Recall@10 = 0.25
    Higher is better, max value = 1.0
    """
    # Cannot recall anything without relevant movies
    if len(relevant) == 0:
        return 0.0
    
    # Consider only top K recommendations
    top_k = recommended[:k]
    
    # Count relevant movies we caught in top K
    hits = len([m for m in top_k if m in relevant])
    
    # Divide by total relevant movies — how much of relevant set did we cover
    return hits / len(relevant)


def ndcg_at_k(recommended, relevant, k):
    """
    NDCG@K: measures ranking quality — rewards putting relevant movies at top.
    
    Question: did we rank movies user liked near the TOP of our list?
    A relevant movie at position 1 contributes more than at position 10.
    Score 1.0 = perfect ranking, 0.0 = no relevant movies in top K
    Higher is better, max value = 1.0
    """
    top_k = recommended[:k]
    
    # DCG: sum discounted gains — higher positions get more credit
    # log2(i+2): position 1 → 1.58, position 10 → 3.58 (larger = less credit)
    dcg = sum([
        1.0 / np.log2(i + 2)
        for i, m in enumerate(top_k)
        if m in relevant
    ])
    
    # IDCG: ideal DCG if all relevant movies ranked at very top
    ideal_hits = min(len(relevant), k)
    idcg = sum([1.0 / np.log2(i + 2) for i in range(ideal_hits)])
    
    # Normalize by ideal to get score between 0 and 1
    return dcg / idcg if idcg > 0 else 0.0


print("Evaluation metric functions defined!")
print("\nMetric summary:")
print(f"  Precision@K — what fraction of K recommendations are relevant?")
print(f"  Recall@K    — what fraction of all relevant movies did we recommend?")
print(f"  NDCG@K      — did we rank relevant movies near the top?")

Evaluation metric functions defined!

Metric summary:
  Precision@K — what fraction of K recommendations are relevant?
  Recall@K    — what fraction of all relevant movies did we recommend?
  NDCG@K      — did we rank relevant movies near the top?


# Score Functions
These are the same scoring functions from Phase 6, consolidated here so this notebook is self contained. Each function returns a dictionary of {movie_id: score} which makes it easy to pass to our evaluation function regardless of which model generated the scores.

In [5]:
def get_svd_scores(user_id, model, movies_df, rated_movie_ids):
    """
    Get SVD predicted ratings for all unrated movies.
    SVD inference is fast — just a dot product of pre-learned vectors.
    Returns dict: {movie_id: predicted_rating}
    """
    all_movie_ids = movies_df['movie_id'].unique()
    
    # Filter to unrated movies only before predicting
    unrated_movies = [m for m in all_movie_ids if m not in rated_movie_ids]
    
    # Build score dictionary — .est is Surprise's estimated rating
    scores = {
        movie_id: model.predict(user_id, movie_id).est
        for movie_id in unrated_movies
    }
    return scores


def get_ncf_scores(user_id, model, movies_df, rated_movie_ids,
                   user_encoder, movie_encoder):
    """
    Get NCF predicted ratings for all unrated movies.
    Uses batch prediction — all movies predicted in one call.
    Returns dict: {movie_id: predicted_rating}
    """
    # Cannot predict for users not seen during training
    if user_id not in user_encoder.classes_:
        return {}
    
    # Encode user ID to integer index
    user_idx = user_encoder.transform([user_id])[0]
    
    all_movie_ids = movies_df['movie_id'].unique()
    
    # Filter to unrated movies that exist in encoder
    unrated_movies = [
        m for m in all_movie_ids
        if m not in rated_movie_ids
        and m in movie_encoder.classes_
    ]
    
    # Encode movie IDs to integer indices
    movie_indices_arr = movie_encoder.transform(unrated_movies)
    
    # Repeat user index for every movie — model needs (user, movie) pairs
    user_arr = np.array([user_idx] * len(unrated_movies))
    movie_arr = np.array(movie_indices_arr)
    
    # Batch prediction — all movies scored in one efficient call
    predictions = model.predict(
        [user_arr, movie_arr],
        batch_size=512,
        verbose=0
    ).flatten()
    
    # Reverse normalization from 0-1 back to 1-5 scale
    predictions = predictions * 4 + 1
    predictions = np.clip(predictions, 1, 5)
    
    return dict(zip(unrated_movies, predictions))


def get_content_scores(user_id, train_df, movies_df,
                       cosine_sim, movie_indices, rated_movie_ids):
    """
    Get accumulated content based similarity scores for all unrated movies.
    Scores are accumulated cosine similarity — not on 1-5 scale.
    Returns dict: {movie_id: accumulated_similarity_score}
    """
    # Only use movies user rated highly as positive signals
    user_ratings = train_df[
        (train_df['user_id'] == user_id) &
        (train_df['rating'] >= 4)
    ].copy()
    
    if user_ratings.empty:
        return {}
    
    # Merge to get movie titles for similarity matrix lookup
    user_ratings = pd.merge(
        user_ratings[['user_id', 'movie_id', 'rating']],
        movies_df[['movie_id', 'title']],
        on='movie_id',
        how='inner'
    )
    
    # Initialize empty score array — one slot per movie
    scores = np.zeros(len(movies_df))
    
    # Accumulate similarity scores from each highly rated movie
    for _, row in user_ratings.iterrows():
        title = row['title']
        if title in movie_indices:
            idx = movie_indices[title]
            scores += cosine_sim[idx]
    
    # Convert to dictionary — only include unrated movies
    result = {}
    for i, movie_id in enumerate(movies_df['movie_id'].values):
        if movie_id not in rated_movie_ids:
            result[movie_id] = scores[i]
    
    return result


def get_hybrid_scores(user_id, movies_df, rated_movie_ids,
                      svd_weight=0.4, ncf_weight=0.4, content_weight=0.2):
    """
    Get normalized hybrid scores combining all three models.
    MinMaxScaler normalizes all scores to 0-1 before combining.
    Returns dict: {movie_id: hybrid_score}
    """
    # Get raw scores from each model independently
    svd_scores = get_svd_scores(user_id, svd_model, movies_df, rated_movie_ids)
    ncf_scores = get_ncf_scores(user_id, ncf_model, movies_df, rated_movie_ids,
                                 user_encoder, movie_encoder)
    content_scores = get_content_scores(user_id, train, movies_df,
                                         cosine_sim, movie_indices, rated_movie_ids)
    
    # Intersection — only movies all three models can score
    common_movies = set(svd_scores.keys()) & \
                    set(ncf_scores.keys()) & \
                    set(content_scores.keys())
    
    if not common_movies:
        return {}
    
    # Sort for consistent array alignment
    movie_list = sorted(list(common_movies))
    
    # Extract scores in same order
    svd_arr = np.array([svd_scores[m] for m in movie_list]).reshape(-1, 1)
    ncf_arr = np.array([ncf_scores[m] for m in movie_list]).reshape(-1, 1)
    content_arr = np.array([content_scores[m] for m in movie_list]).reshape(-1, 1)
    
    # Normalize all scores to 0-1 — fixes scale mismatch problem
    # SVD/NCF: 1-5 scale, Content: 0-382 scale → all become 0-1
    scaler = MinMaxScaler()
    svd_norm = scaler.fit_transform(svd_arr).flatten()
    ncf_norm = scaler.fit_transform(ncf_arr).flatten()
    content_norm = scaler.fit_transform(content_arr).flatten()
    
    # Weighted combination
    hybrid = (svd_weight * svd_norm +
              ncf_weight * ncf_norm +
              content_weight * content_norm)
    
    return dict(zip(movie_list, hybrid))


print("All scoring functions defined!")

All scoring functions defined!


# Evaluation Function
The evaluation simulates real deployment — we pretend the model has only seen training data, generate recommendations, then check against test data to see how many relevant movies were successfully surfaced. Using the same 100 randomly sampled users for all four models ensures the comparison is completely fair — every model is evaluated on identical users with identical ground truth.

In [6]:
def evaluate_model(model_name, score_fn, test_df, train_df,
                   movies_df, k=10, n_users=100):
    """
    Evaluate a recommendation model using Precision@K, Recall@K and NDCG@K.
    
    Logic:
    1. Sample n_users from users present in both train and test
    2. For each user get their highly rated movies in test (ground truth)
    3. Get model recommendations based on train history
    4. Measure how many ground truth movies appear in top K
    5. Average metrics across all users
    """
    precisions = []
    recalls = []
    ndcgs = []
    
    # Find users present in both train and test
    train_users = set(train_df['user_id'].unique())
    test_users = set(test_df['user_id'].unique())
    common_users = list(train_users & test_users)
    
    # Sample n_users with fixed seed for reproducibility
    # Same users evaluated for every model — ensures fair comparison
    np.random.seed(42)
    sample_users = np.random.choice(
        common_users,
        size=min(n_users, len(common_users)),
        replace=False
    )
    
    print(f"\nEvaluating {model_name} on {len(sample_users)} users...")
    
    for i, user_id in enumerate(sample_users):
        
        # Progress update every 10 users
        if (i + 1) % 10 == 0:
            print(f"  Progress: {i+1}/{len(sample_users)} users done")
        
        # Ground truth — movies user rated >= 4 in TEST set
        # These are movies we hope our model recommends
        relevant_movies = set(
            test_df[
                (test_df['user_id'] == user_id) &
                (test_df['rating'] >= 4)
            ]['movie_id'].values
        )
        
        # Skip users with no highly rated movies in test set
        if len(relevant_movies) == 0:
            continue
        
        # Movies seen in training — exclude from recommendations
        rated_movie_ids = set(
            train_df[train_df['user_id'] == user_id]['movie_id'].values
        )
        
        # Get scores from model
        scores = score_fn(user_id, movies_df, rated_movie_ids)
        
        if not scores:
            continue
        
        # Sort by score descending to get ranked recommendation list
        recommended = sorted(
            scores.keys(),
            key=lambda x: scores[x],
            reverse=True
        )
        
        # Calculate all three metrics for this user
        p = precision_at_k(recommended, relevant_movies, k)
        r = recall_at_k(recommended, relevant_movies, k)
        n = ndcg_at_k(recommended, relevant_movies, k)
        
        precisions.append(p)
        recalls.append(r)
        ndcgs.append(n)
    
    # Average metrics across all evaluated users
    results = {
        'model': model_name,
        f'precision@{k}': round(np.mean(precisions), 4),
        f'recall@{k}': round(np.mean(recalls), 4),
        f'ndcg@{k}': round(np.mean(ndcgs), 4),
        'k': k,
        'n_users_evaluated': len(precisions)
    }
    
    print(f"  {model_name} done!")
    print(f"  Precision@{k}: {results[f'precision@{k}']}")
    print(f"  Recall@{k}:    {results[f'recall@{k}']}")
    print(f"  NDCG@{k}:      {results[f'ndcg@{k}']}")
    
    return results


print("Evaluation function defined!")

Evaluation function defined!


# Run Evaluation

In [7]:
# K=10 — evaluate top 10 recommendations per user
# This is standard in recommendation system research
K = 10

# N_USERS=100 — sample 100 users for evaluation
# SVD scoring is slow (one prediction per movie per user)
# 100 users gives statistically meaningful results in reasonable time
N_USERS = 100

print("=" * 60)
print(f"PHASE 1: Evaluating all models @ K={K} on {N_USERS} users")
print(f"Original hybrid weights: {hybrid_config}")
print("=" * 60)

# --- Evaluate SVD ---
# Lambda wraps get_svd_scores to match evaluate_model's expected signature
# evaluate_model calls score_fn(user_id, movies_df, rated_movie_ids)
svd_score_fn = lambda user_id, movies_df, rated: get_svd_scores(
    user_id, svd_model, movies_df, rated
)
svd_results = evaluate_model(
    "SVD", svd_score_fn, test, train, movies, k=K, n_users=N_USERS
)

# --- Evaluate NCF ---
ncf_score_fn = lambda user_id, movies_df, rated: get_ncf_scores(
    user_id, ncf_model, movies_df, rated,
    user_encoder, movie_encoder
)
ncf_results = evaluate_model(
    "NCF", ncf_score_fn, test, train, movies, k=K, n_users=N_USERS
)

# --- Evaluate Content Based ---
content_score_fn = lambda user_id, movies_df, rated: get_content_scores(
    user_id, train, movies_df,
    cosine_sim, movie_indices, rated
)
content_results = evaluate_model(
    "Content Based", content_score_fn, test, train, movies, k=K, n_users=N_USERS
)

# --- Evaluate Hybrid ---
# Uses weights from hybrid_config.json saved in Phase 6
hybrid_score_fn = lambda user_id, movies_df, rated: get_hybrid_scores(
    user_id, movies_df, rated,
    svd_weight=hybrid_config['svd_weight'],
    ncf_weight=hybrid_config['ncf_weight'],
    content_weight=hybrid_config['content_weight']
)
hybrid_results = evaluate_model(
    "Hybrid", hybrid_score_fn, test, train, movies, k=K, n_users=N_USERS
)

print("\n" + "=" * 60)
print("All evaluations complete!")

PHASE 1: Evaluating all models @ K=10 on 100 users
Original hybrid weights: {'svd_weight': 0.4, 'ncf_weight': 0.4, 'content_weight': 0.2}

Evaluating SVD on 100 users...
  Progress: 10/100 users done
  Progress: 20/100 users done
  Progress: 30/100 users done
  Progress: 40/100 users done
  Progress: 50/100 users done
  Progress: 60/100 users done
  Progress: 70/100 users done
  Progress: 80/100 users done
  Progress: 90/100 users done
  Progress: 100/100 users done
  SVD done!
  Precision@10: 0.0837
  Recall@10:    0.0167
  NDCG@10:      0.0747

Evaluating NCF on 100 users...
  Progress: 10/100 users done
  Progress: 20/100 users done
  Progress: 30/100 users done
  Progress: 40/100 users done
  Progress: 50/100 users done
  Progress: 60/100 users done
  Progress: 70/100 users done
  Progress: 80/100 users done
  Progress: 90/100 users done
  Progress: 100/100 users done
  NCF done!
  Precision@10: 0.098
  Recall@10:    0.0208
  NDCG@10:      0.097

Evaluating Content Based on 100 use

# Display Results

In [8]:
# Combine all four model results into one dataframe
results_list = [svd_results, ncf_results, content_results, hybrid_results]
results_df = pd.DataFrame(results_list)

print("\n" + "=" * 60)
print(f"ORIGINAL EVALUATION RESULTS @ K={K}")
print("=" * 60)

# Display clean results table
print(results_df[[
    'model', f'precision@{K}', f'recall@{K}', f'ndcg@{K}',
    'n_users_evaluated'
]].to_string(index=False))

# Save original results
results_df.to_csv(f'{base_path}/evaluation_results.csv', index=False)
print("\nOriginal results saved!")


ORIGINAL EVALUATION RESULTS @ K=10
        model  precision@10  recall@10  ndcg@10  n_users_evaluated
          SVD        0.0837     0.0167   0.0747                 98
          NCF        0.0980     0.0208   0.0970                 98
Content Based        0.0153     0.0032   0.0186                 98
       Hybrid        0.0980     0.0256   0.1014                 98

Original results saved!


# Update Hybrid Weights Based on Results

In [9]:
# Based on evaluation results:
# NCF outperforms SVD on all ranking metrics
# Precision@10: NCF=0.098  > SVD=0.0837
# Recall@10:    NCF=0.0208 > SVD=0.0167
# NDCG@10:      NCF=0.097  > SVD=0.0747
# Content Based is significantly weaker on all metrics

# Update weights to reflect actual performance
hybrid_config_updated = {
    'svd_weight': 0.35,     # reduced — NCF outperforms SVD on ranking
    'ncf_weight': 0.50,     # increased — best individual ranking model
    'content_weight': 0.15  # reduced — weakest model by significant margin
}

# Save updated config
with open(f'{base_path}/hybrid_config.json', 'w') as f:
    json.dump(hybrid_config_updated, f)

print("Updated hybrid config saved!")
print(f"\nOld weights: SVD={hybrid_config['svd_weight']}, "
      f"NCF={hybrid_config['ncf_weight']}, "
      f"Content={hybrid_config['content_weight']}")
print(f"New weights: SVD={hybrid_config_updated['svd_weight']}, "
      f"NCF={hybrid_config_updated['ncf_weight']}, "
      f"Content={hybrid_config_updated['content_weight']}")

Updated hybrid config saved!

Old weights: SVD=0.4, NCF=0.4, Content=0.2
New weights: SVD=0.35, NCF=0.5, Content=0.15


# Re-evaluate Hybrid with Updated Weights

In [10]:
print("=" * 60)
print("PHASE 2: Re-evaluating Hybrid with updated weights")
print(f"New weights: {hybrid_config_updated}")
print("=" * 60)

# Score function with updated weights
hybrid_score_fn_updated = lambda user_id, movies_df, rated: get_hybrid_scores(
    user_id, movies_df, rated,
    svd_weight=hybrid_config_updated['svd_weight'],
    ncf_weight=hybrid_config_updated['ncf_weight'],
    content_weight=hybrid_config_updated['content_weight']
)

# Re-evaluate hybrid with new weights
hybrid_results_updated = evaluate_model(
    "Hybrid (Tuned)",
    hybrid_score_fn_updated,
    test, train, movies,
    k=K, n_users=N_USERS
)

PHASE 2: Re-evaluating Hybrid with updated weights
New weights: {'svd_weight': 0.35, 'ncf_weight': 0.5, 'content_weight': 0.15}

Evaluating Hybrid (Tuned) on 100 users...
  Progress: 10/100 users done
  Progress: 20/100 users done
  Progress: 30/100 users done
  Progress: 40/100 users done
  Progress: 50/100 users done
  Progress: 60/100 users done
  Progress: 70/100 users done
  Progress: 80/100 users done
  Progress: 90/100 users done
  Progress: 100/100 users done
  Hybrid (Tuned) done!
  Precision@10: 0.1041
  Recall@10:    0.0277
  NDCG@10:      0.1018


# Compare Original vs Updated Hybrid

In [11]:
print("\n" + "=" * 65)
print("HYBRID WEIGHTS COMPARISON")
print("=" * 65)
print(f"\n{'Config':<28} {'Precision@10':<15} {'Recall@10':<15} {'NDCG@10'}")
print("-" * 65)
print(
    f"{'Original (SVD=0.4,NCF=0.4)':<28} "
    f"{hybrid_results[f'precision@{K}']:<15} "
    f"{hybrid_results[f'recall@{K}']:<15} "
    f"{hybrid_results[f'ndcg@{K}']}"
)
print(
    f"{'Tuned (SVD=0.35,NCF=0.5)':<28} "
    f"{hybrid_results_updated[f'precision@{K}']:<15} "
    f"{hybrid_results_updated[f'recall@{K}']:<15} "
    f"{hybrid_results_updated[f'ndcg@{K}']}"
)
print("=" * 65)

# Determine which config performs better overall
old_avg = np.mean([
    hybrid_results[f'precision@{K}'],
    hybrid_results[f'recall@{K}'],
    hybrid_results[f'ndcg@{K}']
])
new_avg = np.mean([
    hybrid_results_updated[f'precision@{K}'],
    hybrid_results_updated[f'recall@{K}'],
    hybrid_results_updated[f'ndcg@{K}']
])

print(f"\nOriginal config average: {old_avg:.4f}")
print(f"Tuned config average:    {new_avg:.4f}")

if new_avg > old_avg:
    print("\nTuned weights perform better — keeping updated config ✅")
    best_hybrid_results = hybrid_results_updated
    best_hybrid_label = "Hybrid (Tuned)"
else:
    print("\nOriginal weights perform better — reverting to original config")
    # Revert config file to original weights
    with open(f'{base_path}/hybrid_config.json', 'w') as f:
        json.dump(hybrid_config, f)
    print("Reverted to original config!")
    best_hybrid_results = hybrid_results
    best_hybrid_label = "Hybrid"


HYBRID WEIGHTS COMPARISON

Config                       Precision@10    Recall@10       NDCG@10
-----------------------------------------------------------------
Original (SVD=0.4,NCF=0.4)   0.098           0.0256          0.1014
Tuned (SVD=0.35,NCF=0.5)     0.1041          0.0277          0.1018

Original config average: 0.0750
Tuned config average:    0.0779

Tuned weights perform better — keeping updated config ✅


# Final Complete Comparision Table

In [12]:
print("\n" + "=" * 78)
print("FINAL COMPLETE MODEL COMPARISON")
print("=" * 78)
print(
    f"\n{'Model':<22} {'RMSE':<10} {'MAE':<10} "
    f"{'P@10':<10} {'R@10':<10} {'NDCG@10':<10}"
)
print("-" * 78)

# All four models with all available metrics
models_final = [
    ('SVD',           '0.9372', '0.7440', svd_results),
    ('Neural CF',     '0.9587', '0.7598', ncf_results),
    ('Content Based', 'N/A',    'N/A',    content_results),
    (best_hybrid_label, 'N/A',  'N/A',   best_hybrid_results)
]

for model_name, rmse, mae, results in models_final:
    print(
        f"{model_name:<22} "
        f"{rmse:<10} "
        f"{mae:<10} "
        f"{results[f'precision@{K}']:<10} "
        f"{results[f'recall@{K}']:<10} "
        f"{results[f'ndcg@{K}']:<10}"
    )

print("=" * 78)
print("\nNote:")
print("  RMSE/MAE   — rating prediction accuracy (lower is better)")
print("  P@10/R@10/NDCG@10 — recommendation quality (higher is better)")
print("  N/A — metric not applicable for this model type")


FINAL COMPLETE MODEL COMPARISON

Model                  RMSE       MAE        P@10       R@10       NDCG@10   
------------------------------------------------------------------------------
SVD                    0.9372     0.7440     0.0837     0.0167     0.0747    
Neural CF              0.9587     0.7598     0.098      0.0208     0.097     
Content Based          N/A        N/A        0.0153     0.0032     0.0186    
Hybrid (Tuned)         N/A        N/A        0.1041     0.0277     0.1018    

Note:
  RMSE/MAE   — rating prediction accuracy (lower is better)
  P@10/R@10/NDCG@10 — recommendation quality (higher is better)
  N/A — metric not applicable for this model type


# Save all Results

In [13]:
# Save complete final results
complete_results = pd.DataFrame([
    {
        'model': 'SVD',
        'rmse': 0.9372,
        'mae': 0.7440,
        f'precision@{K}': svd_results[f'precision@{K}'],
        f'recall@{K}': svd_results[f'recall@{K}'],
        f'ndcg@{K}': svd_results[f'ndcg@{K}']
    },
    {
        'model': 'Neural CF',
        'rmse': 0.9587,
        'mae': 0.7598,
        f'precision@{K}': ncf_results[f'precision@{K}'],
        f'recall@{K}': ncf_results[f'recall@{K}'],
        f'ndcg@{K}': ncf_results[f'ndcg@{K}']
    },
    {
        'model': 'Content Based',
        'rmse': None,
        'mae': None,
        f'precision@{K}': content_results[f'precision@{K}'],
        f'recall@{K}': content_results[f'recall@{K}'],
        f'ndcg@{K}': content_results[f'ndcg@{K}']
    },
    {
        'model': best_hybrid_label,
        'rmse': None,
        'mae': None,
        f'precision@{K}': best_hybrid_results[f'precision@{K}'],
        f'recall@{K}': best_hybrid_results[f'recall@{K}'],
        f'ndcg@{K}': best_hybrid_results[f'ndcg@{K}']
    }
])

complete_results.to_csv(f'{base_path}/final_evaluation_results.csv', index=False)
print("Final evaluation results saved to final_evaluation_results.csv!")
print("\nAll evaluation complete!")

Final evaluation results saved to final_evaluation_results.csv!

All evaluation complete!
